> **IMPORTANTE:** Este notebook requiere el perfil **completo** de Docker.
> Si aun no lo has levantado, ejecuta en tu terminal:
> ```bash
> docker-compose --profile completo up -d
> ```
> Espera ~1 minuto para que Hive Metastore inicie completamente.

# EA2 - Actividad 2.3b: Hive Metastore con Spark

## Objetivos
- Conectar Spark a Hive Metastore
- Crear tablas persistentes y externas
- Entender la diferencia entre tablas gestionadas y externas
- Aplicar particionamiento de datos
- Ejecutar consultas SQL sobre tablas Hive

## Conceptos Clave

### Apache Hive como Data Warehouse

Apache Hive es un sistema de data warehouse construido sobre Hadoop que permite consultar grandes volumenes de datos usando SQL. El componente central es el **Metastore**, que almacena los metadatos de las tablas (schema, ubicacion, formato, particiones).

### Hive Metastore

El Metastore es un servicio centralizado que almacena:
- **Bases de datos** y sus ubicaciones
- **Esquemas de tablas** (columnas, tipos de datos)
- **Ubicacion de los datos** en el sistema de archivos
- **Informacion de particiones**

Spark se conecta al Metastore via el protocolo **Thrift** para leer y escribir esta metadata.

### Tablas Gestionadas vs Externas

| Caracteristica | Tabla Gestionada (Managed) | Tabla Externa (External) |
|----------------|---------------------------|-------------------------|
| Datos | Spark gestiona los datos | Los datos existen externamente |
| DROP TABLE | Elimina datos + metadata | Solo elimina metadata |
| Uso tipico | Datos procesados internamente | Datos compartidos entre herramientas |
| Comando | `saveAsTable()` | `CREATE EXTERNAL TABLE` |

### Apache Parquet

Parquet es un formato **columnar** optimizado para Big Data:
- Compresion eficiente por columna
- Lectura selectiva de columnas (projection pushdown)
- Schema embebido en el archivo
- Formato estandar en el ecosistema Hadoop/Spark

### Particionamiento

El particionamiento divide los datos en subdirectorios basados en el valor de una o mas columnas:
```
vuelos_por_mes/
  MONTH=1/
    part-00000.parquet
  MONTH=2/
    part-00000.parquet
  ...
```
Esto acelera las consultas que filtran por la columna de particion, ya que Spark solo lee los directorios relevantes.

## Setup

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("hive_metastore") \
    .master("local[*]") \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print("Hive support habilitado")

## 1. Verificar Conexion al Metastore

Lo primero es verificar que Spark puede comunicarse con el Hive Metastore. Si este comando falla, asegurate de que el servicio `hive-metastore` este corriendo.

In [ ]:
# Verificar conexion listando las bases de datos disponibles
spark.sql("SHOW DATABASES").show()

## 2. Crear Base de Datos

Las bases de datos en Hive son contenedores logicos para organizar tablas. Crearemos una base de datos llamada `bigdata` para nuestro proyecto.

In [ ]:
# Crear base de datos (IF NOT EXISTS evita error si ya existe)
spark.sql("CREATE DATABASE IF NOT EXISTS bigdata")

# Verificar que se creo
spark.sql("SHOW DATABASES").show()

# Usar la base de datos
spark.sql("USE bigdata")
print("Base de datos 'bigdata' seleccionada")

## 3. Crear Tabla Gestionada desde DataFrame

Una **tabla gestionada** (managed table) es aquella donde Spark controla tanto los datos como los metadatos. Al usar `saveAsTable()`, Spark escribe los datos en el warehouse de Hive y registra la tabla en el Metastore.

In [ ]:
# Leer datos de vuelos
df = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True)

print(f"Registros cargados: {df.count()}")
df.printSchema()

In [ ]:
# Guardar como tabla gestionada en Hive
df.write.mode("overwrite").saveAsTable("bigdata.vuelos")

# Verificar que la tabla existe
spark.sql("SHOW TABLES IN bigdata").show()

# Consultar la tabla
spark.sql("SELECT COUNT(*) FROM bigdata.vuelos").show()

In [ ]:
# Ver informacion detallada de la tabla
spark.sql("DESCRIBE EXTENDED bigdata.vuelos").show(truncate=False)

## 4. Crear Tabla Externa apuntando a Parquet

Una **tabla externa** no gestiona los datos; solo apunta a una ubicacion existente. Al eliminar la tabla, los datos permanecen intactos.

Primero guardamos datos en formato Parquet, y luego creamos una tabla externa que apunta a esa ubicacion.

In [ ]:
# Guardar un subconjunto de datos como Parquet
df.select("YEAR", "MONTH", "DAY", "AIRLINE", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DEPARTURE_DELAY", "ARRIVAL_DELAY", "DISTANCE") \
    .write.mode("overwrite") \
    .parquet("/home/jovyan/datos/vuelos_parquet")

print("Datos guardados en formato Parquet")

In [ ]:
# Crear tabla externa apuntando a los archivos Parquet
spark.sql("""
    CREATE EXTERNAL TABLE IF NOT EXISTS bigdata.vuelos_ext (
        YEAR INT,
        MONTH INT,
        DAY INT,
        AIRLINE STRING,
        ORIGIN_AIRPORT STRING,
        DESTINATION_AIRPORT STRING,
        DEPARTURE_DELAY DOUBLE,
        ARRIVAL_DELAY DOUBLE,
        DISTANCE DOUBLE
    )
    STORED AS PARQUET
    LOCATION '/home/jovyan/datos/vuelos_parquet'
""")

# Verificar
spark.sql("SELECT COUNT(*) as total FROM bigdata.vuelos_ext").show()
spark.sql("SELECT * FROM bigdata.vuelos_ext LIMIT 5").show()

## 5. Consultas SQL sobre Tablas Hive

Una vez que las tablas estan registradas en el Metastore, podemos ejecutar consultas SQL directamente sobre ellas, como si fueran tablas en una base de datos tradicional.

In [ ]:
# Listar todas las tablas disponibles
spark.sql("SHOW TABLES IN bigdata").show()

In [ ]:
# Consulta: conteo de vuelos por aerolinea
spark.sql("""
    SELECT AIRLINE, COUNT(*) as total_vuelos
    FROM bigdata.vuelos
    GROUP BY AIRLINE
    ORDER BY total_vuelos DESC
""").show()

In [ ]:
# Consulta: promedio de retraso por mes
spark.sql("""
    SELECT MONTH, 
           ROUND(AVG(DEPARTURE_DELAY), 2) as avg_retraso_salida,
           ROUND(AVG(ARRIVAL_DELAY), 2) as avg_retraso_llegada
    FROM bigdata.vuelos
    WHERE DEPARTURE_DELAY IS NOT NULL AND ARRIVAL_DELAY IS NOT NULL
    GROUP BY MONTH
    ORDER BY MONTH
""").show()

In [ ]:
# Consulta: top 10 rutas mas frecuentes
spark.sql("""
    SELECT ORIGIN_AIRPORT, DESTINATION_AIRPORT, 
           COUNT(*) as total_vuelos
    FROM bigdata.vuelos_ext
    GROUP BY ORIGIN_AIRPORT, DESTINATION_AIRPORT
    ORDER BY total_vuelos DESC
    LIMIT 10
""").show()

## 6. Particionamiento de Tablas

El particionamiento organiza los datos en subdirectorios segun el valor de una columna. Esto mejora drasticamente el rendimiento de consultas que filtran por esa columna, ya que Spark solo necesita leer las particiones relevantes (**partition pruning**).

In [ ]:
# Crear tabla particionada por MONTH
df.write.mode("overwrite").partitionBy("MONTH").saveAsTable("bigdata.vuelos_por_mes")

print("Tabla particionada creada exitosamente")

In [ ]:
# Ver las particiones disponibles
spark.sql("SHOW PARTITIONS bigdata.vuelos_por_mes").show()

In [ ]:
# Consulta optimizada: solo lee la particion MONTH=1
# Gracias al partition pruning, Spark no lee los datos de otros meses
spark.sql("""
    SELECT AIRLINE, COUNT(*) as vuelos_enero
    FROM bigdata.vuelos_por_mes
    WHERE MONTH = 1
    GROUP BY AIRLINE
    ORDER BY vuelos_enero DESC
""").show()

In [ ]:
# Comparar: descripcion de tabla particionada vs no particionada
print("=== Tabla NO particionada ===")
spark.sql("DESCRIBE EXTENDED bigdata.vuelos").show(truncate=False)

print("\n=== Tabla particionada ===")
spark.sql("DESCRIBE EXTENDED bigdata.vuelos_por_mes").show(truncate=False)

---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [ ]:
# =============================================================
# EJERCICIO 1: Crear tabla particionada por AIRLINE
# =============================================================

# 1. Guardar como tabla particionada por AIRLINE
df.write.mode("overwrite").partitionBy("AIRLINE").saveAsTable("bigdata.vuelos_particionada")
print("Tabla bigdata.vuelos_particionada creada exitosamente")

# 2. Verificar las particiones
print("\nParticiones disponibles:")
spark.sql("SHOW PARTITIONS bigdata.vuelos_particionada").show(truncate=False)

# 3. Consulta filtrando por una aerolinea especifica
print("\nVuelos de la aerolinea AA (American Airlines):")
spark.sql("""
    SELECT AIRLINE, ORIGIN_AIRPORT, DESTINATION_AIRPORT, COUNT(*) as total_vuelos
    FROM bigdata.vuelos_particionada
    WHERE AIRLINE = 'AA'
    GROUP BY AIRLINE, ORIGIN_AIRPORT, DESTINATION_AIRPORT
    ORDER BY total_vuelos DESC
    LIMIT 10
""").show()

> **Nota docente:** El particionamiento por AIRLINE crea un subdirectorio por cada
> aerolinea. Esto es eficiente porque:
> - Hay un numero limitado de aerolineas (~14), lo que genera pocas particiones.
> - Las consultas que filtran por aerolinea solo leen la particion relevante.
>
> Regla practica: particionar por columnas con **baja cardinalidad** (pocos valores unicos).
> Particionar por una columna con miles de valores (como ORIGIN_AIRPORT) crearia
> demasiados subdirectorios pequenos, lo que degradaria el rendimiento.
>
> Error comun: olvidar que `SHOW PARTITIONS` solo funciona con tablas Hive particionadas,
> no con vistas temporales ni con tablas sin particionar.

In [ ]:
# =============================================================
# EJERCICIO 2: Consultas SQL sobre tablas Hive
# =============================================================

# Query 1: Conteo de vuelos por mes, ordenado por mes
print("=== Query 1: Vuelos por mes ===")
spark.sql("""
    SELECT MONTH, COUNT(*) as total_vuelos
    FROM bigdata.vuelos_por_mes
    GROUP BY MONTH
    ORDER BY MONTH
""").show()

# Query 2: Promedio de retraso de salida por aerolinea
print("=== Query 2: Promedio retraso por aerolinea ===")
spark.sql("""
    SELECT AIRLINE, ROUND(AVG(DEPARTURE_DELAY), 2) as avg_retraso
    FROM bigdata.vuelos_particionada
    WHERE DEPARTURE_DELAY IS NOT NULL
    GROUP BY AIRLINE
    ORDER BY avg_retraso DESC
""").show()

# Query 3: Top 5 rutas con mas vuelos
print("=== Query 3: Top 5 rutas ===")
spark.sql("""
    SELECT ORIGIN_AIRPORT, DESTINATION_AIRPORT, COUNT(*) as total
    FROM bigdata.vuelos
    GROUP BY ORIGIN_AIRPORT, DESTINATION_AIRPORT
    ORDER BY total DESC
    LIMIT 5
""").show()

> **Nota docente:** Las tres queries demuestran que las tablas Hive se consultan
> exactamente igual que vistas temporales. La diferencia clave es la **persistencia**:
> - Vistas temporales desaparecen al cerrar la SparkSession.
> - Tablas Hive persisten entre sesiones y son visibles para otros usuarios/herramientas.
>
> Notar que Query 1 usa la tabla particionada por mes, Query 2 la particionada por
> aerolinea, y Query 3 la tabla sin particionar. En la practica, elegir la tabla
> correcta segun el filtro de la consulta.

---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Disenar un schema de data warehouse para datos de ventas.

In [ ]:
# =============================================================
# DESAFIO: Disenar un Data Warehouse en Hive
# =============================================================

from pyspark.sql import functions as F

# a) Leer los datos fuente
df_sales = spark.read.csv("/home/jovyan/datos/sales.csv", header=True, inferSchema=True)
df_stores = spark.read.csv("/home/jovyan/datos/stores.csv", header=True, inferSchema=True)

print(f"Sales: {df_sales.count()} registros")
print(f"Stores: {df_stores.count()} registros")

# b) Crear base de datos para el warehouse
spark.sql("CREATE DATABASE IF NOT EXISTS bigdata_warehouse")
spark.sql("USE bigdata_warehouse")
print("\nBase de datos 'bigdata_warehouse' creada y seleccionada")

In [ ]:
# c) Crear tabla de DIMENSION: dim_tiendas
df_stores.write.mode("overwrite").saveAsTable("bigdata_warehouse.dim_tiendas")
print("Tabla dim_tiendas creada")
spark.sql("SELECT * FROM bigdata_warehouse.dim_tiendas").show()

# d) Crear tabla de DIMENSION: dim_tiempo
# Extraer componentes de fecha para facilitar analisis temporal
df_tiempo = df_sales.select("Date").distinct() \
    .withColumn("fecha", F.to_date("Date", "yyyy-MM-dd")) \
    .withColumn("anio", F.year("fecha")) \
    .withColumn("mes", F.month("fecha")) \
    .withColumn("dia", F.dayofmonth("fecha")) \
    .withColumn("dia_semana", F.dayofweek("fecha")) \
    .withColumn("semana_anio", F.weekofyear("fecha")) \
    .drop("fecha")

df_tiempo.write.mode("overwrite").saveAsTable("bigdata_warehouse.dim_tiempo")
print("\nTabla dim_tiempo creada")
spark.sql("SELECT * FROM bigdata_warehouse.dim_tiempo ORDER BY Date LIMIT 10").show()

In [ ]:
# e) Crear tabla de HECHOS: fact_ventas (particionada por IsHoliday)
df_fact = df_sales.select("Store", "Dept", "Date", "Weekly_Sales", "IsHoliday")
df_fact.write.mode("overwrite").partitionBy("IsHoliday").saveAsTable("bigdata_warehouse.fact_ventas")
print("Tabla fact_ventas creada (particionada por IsHoliday)")

# Verificar particiones
spark.sql("SHOW PARTITIONS bigdata_warehouse.fact_ventas").show()

# Verificar todas las tablas del warehouse
print("\nTablas en bigdata_warehouse:")
spark.sql("SHOW TABLES IN bigdata_warehouse").show()

In [ ]:
# f) Query de ejemplo: ventas totales por tipo de tienda
print("=== Ventas totales por tipo de tienda ===")
spark.sql("""
    SELECT 
        d.Type as tipo_tienda,
        COUNT(*) as num_registros,
        ROUND(SUM(f.Weekly_Sales), 2) as total_ventas,
        ROUND(AVG(f.Weekly_Sales), 2) as promedio_ventas
    FROM bigdata_warehouse.fact_ventas f
    JOIN bigdata_warehouse.dim_tiendas d ON f.Store = d.Store
    GROUP BY d.Type
    ORDER BY total_ventas DESC
""").show()

# g) Query: ventas por mes (usando dim_tiempo)
print("=== Ventas por mes ===")
spark.sql("""
    SELECT 
        t.anio,
        t.mes,
        ROUND(SUM(f.Weekly_Sales), 2) as total_ventas,
        COUNT(*) as num_registros
    FROM bigdata_warehouse.fact_ventas f
    JOIN bigdata_warehouse.dim_tiempo t ON f.Date = t.Date
    GROUP BY t.anio, t.mes
    ORDER BY t.anio, t.mes
""").show(20)

# h) Query: comparar ventas en dias festivos vs normales
print("=== Ventas festivos vs normales ===")
spark.sql("""
    SELECT 
        f.IsHoliday,
        COUNT(*) as num_registros,
        ROUND(AVG(f.Weekly_Sales), 2) as promedio_ventas,
        ROUND(SUM(f.Weekly_Sales), 2) as total_ventas
    FROM bigdata_warehouse.fact_ventas f
    GROUP BY f.IsHoliday
""").show()

> **Nota docente:** Este desafio introduce el concepto de **modelado dimensional**
> (star schema), fundamental en data warehousing:
>
> **Star Schema:**
> - **Tabla de hechos (fact_ventas):** Contiene las metricas medibles (Weekly_Sales)
>   y las claves foraneas a las dimensiones (Store, Date).
> - **Tablas de dimensiones (dim_tiendas, dim_tiempo):** Contienen atributos
>   descriptivos para filtrar y agrupar.
>
> Decisiones de diseno:
> - `dim_tiempo` se construye extrayendo componentes de fecha, lo cual facilita
>   consultas como "ventas por mes" o "ventas por dia de la semana".
> - `fact_ventas` se particiona por `IsHoliday` (booleano, solo 2 valores), lo cual
>   es ideal para consultas que comparan festivos vs normales.
> - En un DW real, se agregarian mas dimensiones (dim_departamento, dim_producto, etc.).
>
> Error comun: particionar la tabla de hechos por una columna con alta cardinalidad
> (como Date), lo que crearia demasiadas particiones pequenas.

---
## Resumen

En esta actividad aprendimos:

1. **Hive Metastore:** Servicio centralizado que almacena metadatos de tablas, schemas y particiones
2. **Conexion Spark-Hive:** Usar `enableHiveSupport()` y configurar `hive.metastore.uris`
3. **Bases de datos:** `CREATE DATABASE` para organizar tablas logicamente
4. **Tablas gestionadas:** `saveAsTable()` - Spark controla datos y metadata
5. **Tablas externas:** `CREATE EXTERNAL TABLE` - Solo metadata, datos externos
6. **Formato Parquet:** Formato columnar eficiente para Big Data
7. **Particionamiento:** `partitionBy()` para organizar datos en subdirectorios y optimizar consultas
8. **Consultas SQL:** Queries directas sobre tablas persistentes en Hive

In [ ]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")